In [1]:
import pandas as pd
import glob

# Get all daily FFB excel files
ffb_files = glob.glob('../raw_data/daily_ffb_*.xlsx')

In [24]:
all_ffb_data = []
for file in ffb_files:
    df = pd.read_excel(file, header=1)

    # Convert numeric columns safely
    df = df.apply(pd.to_numeric, errors="coerce")
    
    # Extract year & month from filename (e.g., ..._MMYYYY.xlsx)
    try:
        fname = file.split('_')[-1].split('.')[0]
        year, month = int(fname[4:]), int(fname[2:4])
    except (ValueError, IndexError):
        raise ValueError(f"Could not extract year and month from filename: {file}")
    
    # Build tidy DataFrame
    df = (
        df.rename(columns={df.columns[0]: "Day"})
          .assign(
              Year=year,
              Month=month,
              FFB_Price=lambda d: d[
                  ["North / Utara", "South / Selatan", "Central / Tengah", 
                   "East Coast / Timur", "Sabah", "Sarawak*"]
              ].mean(axis=1),
              Datetime=lambda d: pd.to_datetime(d[["Year", "Month", "Day"]])
          )
          .set_index("Datetime")[["FFB_Price"]]
          .sort_index()
    )
    all_ffb_data.append(df)

# Concatenate all dataframes
if all_ffb_data:
    ffb_df = pd.concat(all_ffb_data)

In [29]:
ffb_df = ffb_df.sort_index()
ffb_df = ffb_df.dropna(subset=['FFB_Price'])
ffb_df

,FFB_Price
Datetime,
2025-05-02,43.370000
2025-05-05,42.488333
2025-05-06,41.706667
2025-05-07,41.608333
2025-05-08,40.706667
...,...
2025-08-20,46.438333
2025-08-21,46.628333
2025-08-22,46.728333


In [33]:
ffb_df_monthly = ffb_df.resample("ME").mean()
ffb_df_monthly.index = ffb_df_monthly.index.to_period("M")

In [34]:
ffb_df.to_csv('../data/ffb_daily.csv')
ffb_df_monthly.to_csv('../data/ffb_monthly.csv')